# Disease Detection Project (based on Sogandi et al., 2024)

این نوت‌بوک پایتون برای اجرای یک پروژه تشخیص بیماری با ترکیب **استخراج قوانین (Apriori)** و **مدل‌های نظارت‌شده** آماده شده است. 

فایل دیتاست را باید از Kaggle دانلود کنید (مثلاً `disease.csv` یا `Training.csv` از دیتاست‌های زیر) و در همان پوشهٔ نوت‌بوک قرار دهید:
- `Disease Prediction Using Machine Learning` (kaushil268) — لینک در پیام پایین.
- یا `Disease Diagnosis Dataset` (s3programmer).

در ادامه کدها را خط‌به‌خط اجرا کنید.

In [ ]:
# 1) نصب نیازمندی‌ها (اگر لازم است)
!pip install pandas numpy scikit-learn mlxtend xgboost seaborn matplotlib joblib --quiet

print('Packages installed (or already present).')

## 2) بارگذاری کتابخانه‌ها و تنظیمات اولیه

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
print('Libraries loaded')

## 3) بارگذاری دیتاست

لطفاً نام فایل CSV دیتاست را در متغیر `DATA_PATH` قرار دهید. اگر از دیتاست `kaushil268` استفاده می‌کنید، معمولاً فایل `Training.csv` یا `dataset.csv` است.

In [ ]:
# Example: set this to the CSV file you downloaded from Kaggle
DATA_PATH = 'disease_dataset.csv'  # <-- نام فایل را اینجا بگذارید

df = pd.read_csv(DATA_PATH)
df.shape


### نگاهی سریع به داده‌ها

In [ ]:
df.head()


## 4) پیش‌پردازش
- پر کردن مقادیر گمشده
- کدگذاری متغیرهای دسته‌ای
- جدا کردن ویژگی‌ها و برچسب

In [ ]:
# Example preprocessing - ممکن است نیاز به تغییر بسته به ساختار دیتاست داشته باشد
df = df.copy()

# اگر ستونِ target با نام 'prognosis' یا 'Disease' است، آن را شناسایی کنید
target_candidates = [c for c in df.columns if c.lower() in ['prognosis','disease','diagnosis','label','target']]
if len(target_candidates)==0:
    print('Warning: target column not found automatically. Please set the target column name manually.')
    TARGET = 'prognosis'  # <-- اگر نیاز بود این را تغییر دهید
else:
    TARGET = target_candidates[0]
    print('Detected target column:', TARGET)

# پر کردن مقادیر گمشده
for col in df.columns:
    if df[col].dtype=='object':
        df[col] = df[col].fillna('unknown')
    else:
        df[col] = df[col].fillna(df[col].median())

# اگر تمام ویژگی‌ها علامتی (binary symptoms) هستند و با 'Yes'/'No' نمایندگی شده‌اند، آن‌ها را به 0/1 تبدیل کنید
for col in df.columns:
    if df[col].dtype=='object' and col!=TARGET:
        # try to detect yes/no
        vals = set(df[col].dropna().unique())
        if vals.issubset({'Yes','No','yes','no','YES','NO'}):
            df[col] = df[col].map(lambda x: 1 if str(x).lower()=='yes' else 0)

# اگر هنوز ستون‌های متنی دارید، کدگذاری کنید
le = LabelEncoder()
if df[TARGET].dtype=='object':
    df[TARGET] = le.fit_transform(df[TARGET])

X = df.drop(columns=[TARGET])
y = df[TARGET]
print('X shape:', X.shape, 'y shape:', y.shape)


## 5) استخراج قوانین (Apriori)
اگر دیتاست حالتِ باینریِ علائم (هر ستون یک علامت، 0/1) دارد، می‌توان Apriori اجرا کرد.

In [ ]:
# Prepare for apriori: ensure X is 0/1 for symptom columns
binarized = X.copy()
for col in binarized.columns:
    if binarized[col].dtype!='int64' and binarized[col].dtype!='float64':
        try:
            binarized[col] = binarized[col].astype(int)
        except:
            # drop non-numeric columns for Apriori
            binarized.drop(columns=[col], inplace=True)

freq_items = apriori(binarized, min_support=0.05, use_colnames=True)
rules = association_rules(freq_items, metric='lift', min_threshold=1)
print('Frequent itemsets:', freq_items.shape)
print('Rules:', rules.shape)
rules.head()


## 6) مدل‌های نظارت‌شده
مدل‌ها را آموزش می‌دهیم و با معیارهای استاندارد ارزیابی می‌کنیم.

In [ ]:
# تقسیم داده
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# اگر ویژگی‌ها عددی نیستند، یکی-هات انکدینگ یا تبدیل مناسب لازم است
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# مدل‌ها
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=200),
    'SVM': SVC(probability=True)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(f'{name} accuracy: {acc:.4f}')
    print(classification_report(y_test, preds))


### اگر XGBoost نصب و در دسترس است، می‌توان آن را نیز اضافه کرد

In [ ]:
try:
    import xgboost as xgb
    xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
    xgb_clf.fit(X_train_scaled, y_train)
    preds = xgb_clf.predict(X_test_scaled)
    print('XGBoost accuracy:', accuracy_score(y_test, preds))
except Exception as e:
    print('XGBoost not available or failed to run:', e)


## 7) اعتبارسنجی متقابل (Cross-validation) و مقایسه مدل‌ها

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    scores = cross_val_score(model, scaler.fit_transform(pd.get_dummies(X)), y, cv=cv, scoring='accuracy')
    print(f'{name} CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}')


## 8) ذخیره مدل
مدل برتر را ذخیره کنید تا بعداً برای پیش‌بینی سریع بارگذاری شود.

In [ ]:
best_model = RandomForestClassifier(n_estimators=200)
best_model.fit(X_train_scaled, y_train)
joblib.dump({'model': best_model, 'scaler': scaler, 'columns': X_train.columns}, 'disease_model.pkl')
print('Model saved to disease_model.pkl')

## Notes and next steps
- بسته به ساختار دیتاست ممکن است لازم باشد ستون‌های غیرمرتبط حذف یا ویژگی‌های جدید ساخته شود.
- برای مسائل چندکلاسهٔ نامتوازن از تکنیک‌هایی مثل SMOTE یا class weights استفاده کنید.
- برای استخراج قوانین پیچیده‌تر، پارامترهای apriori (min_support) و association_rules را تنظیم کنید.
- این نوت‌بوک یک قالب شروع است؛ در صورت تمایل می‌توانم آن را بر اساس دیتاست خاصی که تو دانلود می‌کنی دقیق‌تر تنظیم کنم.